In [0]:
#creating temporary view

from pyspark.sql import functions as F

energy = spark.table(
    "electricity_catalog.electricity_schema.bronze_energy"
)

peak = spark.table(
    "electricity_catalog.electricity_schema.bronze_peak"
)

karnataka_weather = spark.table(
    "electricity_catalog.electricity_schema.bronze_karnataka_weather"
)

maharashtra_weather = spark.table(
    "electricity_catalog.electricity_schema.bronze_maharashtra_weather"
)

print("Energy:", energy.count())
print("Peak:", peak.count())
print("Karnataka weather:", karnataka_weather.count())
print("Maharashtra weather:", maharashtra_weather.count())

In [0]:
#Checcking Columns
print("ENERGY")
print(energy.columns)

print("\nPEAK")
print(peak.columns)

In [0]:


energy = energy.withColumnRenamed("Date", "date")

energy = energy.withColumn(
    "date",
    F.to_date("date")
)

peak = peak.withColumn(
    "date",
    F.to_date("date")
)

In [0]:
#checking schema
energy.printSchema()


In [0]:
peak.printSchema()

In [0]:
# States we want for the first version
states = [
    "Karnataka",
    "Maharashtra"
]

energy_long = energy.select(
    "date",
    *states
)

stack_expression = ", ".join(
    [f"'{state}', `{state}`" for state in states]
)

energy_long = energy_long.selectExpr(
    "date",
    f"""
    stack(
        {len(states)},
        {stack_expression}
    ) AS (state, energy_demand_MU)
    """
)

In [0]:
energy_long.groupBy("state").count().display()

In [0]:
energy_long.limit(10).display()

In [0]:
# States we want for the first version
states = [
    "Karnataka",
    "Maharashtra"
]

peak_long = peak.select(
    "date",
    *states
)

stack_expression = ", ".join(
    [f"'{state}', `{state}`" for state in states]
)

peak_long = peak_long.selectExpr(
    "date",
    f"""
    stack(
        {len(states)},
        {stack_expression}
    ) AS (state, energy_demand_MU)
    """
)

In [0]:
peak_long.limit(10).display()

In [0]:
#checking for the two states count

peak_long.groupBy("state").count().display()

In [0]:
#Joining the two tables 
#We are prefering left join as we want to preserve the preserve the electricity-demand #records even if a matching peak-demand record is missing.
electricity_df = energy_long.join(peak_long, 
                                  on=["date","state"],
                                  how="left")

In [0]:
electricity_df.limit(10).display()

In [0]:
#preparing Karanataka weather dataset
karnataka_weather = karnataka_weather.withColumn(
    "state",
    F.lit("Karnataka")
)

karnataka_weather = karnataka_weather.withColumn(
    "date",
    F.to_date("date")
)

In [0]:
maharashtra_weather = maharashtra_weather.withColumn(
    "state",
    F.lit("Maharashtra")
)

maharashtra_weather = maharashtra_weather.withColumn(
    "date",
    F.to_date("date")
)

In [0]:
#Since aboth tables have the same structure we are using Union
weather_df = karnataka_weather.union(maharashtra_weather)
weather_df.limit(10).display()


In [0]:
#checking count by state
weather_df.groupBy("state").count().display()

In [0]:
silver = electricity_df.join(weather_df, on=["date","state"],how="left")

In [0]:
silver.limit(10).display()

In [0]:
silver_cols = silver.columns
print(silver_cols)

In [0]:
#checking the join quality
silver.select(
    F.count("*").alias("total_rows"),
    F.count("energy_demand_MU").alias("energy_rows"),
    F.count("peak_demand_MW").alias("peak_rows"),
    F.count("2m_temperature_mean").alias("weather_rows")
).display()

In [0]:
#we observe that we have two columns with the same name in the join
#So we now rectrate the electricity_df
electricity = (
    energy_long.alias("e")
    .join(
        peak_long.alias("p"),
        on=["date", "state"],
        how="left"
    )
    .select(
        "date",
        "state",
        F.col("e.energy_demand_MU").alias("energy_demand_MU"),
        F.col("p.peak_demand_MW").alias("peak_demand_MW")
    )
)

In [0]:
karnataka_weather = (
    karnataka_weather
    .withColumn("state", F.lit("Karnataka"))
    .withColumn("date", F.to_date("date"))
)

maharashtra_weather = (
    maharashtra_weather
    .withColumn("state", F.lit("Maharashtra"))
    .withColumn("date", F.to_date("date"))
)

weather = karnataka_weather.unionByName(
    maharashtra_weather
)

In [0]:
print(weather.columns)

In [0]:
silver = (
    electricity.alias("e")
    .join(
        weather.alias("w"),
        on=["date", "state"],
        how="left"
    )
    .select(
        "date",
        "state",

        F.col("e.energy_demand_MU").alias("energy_demand_MU"),
        F.col("e.peak_demand_MW").alias("peak_demand_MW"),

        F.col("w.`10m_u_component_of_wind_min`").alias("wind_u_min"),
        F.col("w.`10m_u_component_of_wind_mean`").alias("wind_u_mean"),
        F.col("w.`10m_u_component_of_wind_max`").alias("wind_u_max"),

        F.col("w.`10m_v_component_of_wind_min`").alias("wind_v_min"),
        F.col("w.`10m_v_component_of_wind_mean`").alias("wind_v_mean"),
        F.col("w.`10m_v_component_of_wind_max`").alias("wind_v_max"),

        F.col("w.`2m_temperature_min`").alias("temperature_min"),
        F.col("w.`2m_temperature_mean`").alias("temperature_mean"),
        F.col("w.`2m_temperature_max`").alias("temperature_max"),

        F.col("w.`2m_dewpoint_temperature_min`").alias("dewpoint_min"),
        F.col("w.`2m_dewpoint_temperature_mean`").alias("dewpoint_mean"),
        F.col("w.`2m_dewpoint_temperature_max`").alias("dewpoint_max"),

        F.col("w.`surface_solar_radiation_downwards_min`").alias("solar_radiation_min"),
        F.col("w.`surface_solar_radiation_downwards_mean`").alias("solar_radiation_mean"),
        F.col("w.`surface_solar_radiation_downwards_max`").alias("solar_radiation_max"),

        F.col("w.`total_cloud_cover_min`").alias("cloud_cover_min"),
        F.col("w.`total_cloud_cover_mean`").alias("cloud_cover_mean"),
        F.col("w.`total_cloud_cover_max`").alias("cloud_cover_max"),

        F.col("w.`utci_min`").alias("utci_min"),
        F.col("w.`utci_mean`").alias("utci_mean"),
        F.col("w.`utci_max`").alias("utci_max")
    )
)

In [0]:
silver.printSchema()

In [0]:
peak = spark.table(
    "electricity_catalog.electricity_schema.bronze_peak"
)

peak = peak.withColumn(
    "date",
    F.to_date("date")
)

print(peak.columns)

In [0]:
states = [
    "Karnataka",
    "Maharashtra"
]

peak_long = peak.select(
    "date",
    *states
)

stack_expression = ", ".join(
    [f"'{state}', `{state}`" for state in states]
)

peak_long = peak_long.selectExpr(
    "date",
    f"""
    stack(
        {len(states)},
        {stack_expression}
    ) AS (state, peak_demand_MW)
    """
)

In [0]:
print(peak_long.columns)

In [0]:
display(peak_long.limit(10))

In [0]:
electricity = (
    energy_long.alias("e")
    .join(
        peak_long.alias("p"),
        on=["date", "state"],
        how="left"
    )
    .select(
        "date",
        "state",
        F.col("e.energy_demand_MU").alias("energy_demand_MU"),
        F.col("p.peak_demand_MW").alias("peak_demand_MW")
    )
)

In [0]:
print(electricity.columns)

In [0]:
display(electricity.limit(10))

In [0]:
silver = (
    electricity.alias("e")
    .join(
        weather.alias("w"),
        on=["date", "state"],
        how="left"
    )
    .select(
        "date",
        "state",

        F.col("e.energy_demand_MU").alias("energy_demand_MU"),
        F.col("e.peak_demand_MW").alias("peak_demand_MW"),

        F.col("w.`10m_u_component_of_wind_min`").alias("wind_u_min"),
        F.col("w.`10m_u_component_of_wind_mean`").alias("wind_u_mean"),
        F.col("w.`10m_u_component_of_wind_max`").alias("wind_u_max"),

        F.col("w.`10m_v_component_of_wind_min`").alias("wind_v_min"),
        F.col("w.`10m_v_component_of_wind_mean`").alias("wind_v_mean"),
        F.col("w.`10m_v_component_of_wind_max`").alias("wind_v_max"),

        F.col("w.`2m_temperature_min`").alias("temperature_min"),
        F.col("w.`2m_temperature_mean`").alias("temperature_mean"),
        F.col("w.`2m_temperature_max`").alias("temperature_max"),

        F.col("w.`2m_dewpoint_temperature_min`").alias("dewpoint_min"),
        F.col("w.`2m_dewpoint_temperature_mean`").alias("dewpoint_mean"),
        F.col("w.`2m_dewpoint_temperature_max`").alias("dewpoint_max"),

        F.col("w.`surface_solar_radiation_downwards_min`").alias("solar_radiation_min"),
        F.col("w.`surface_solar_radiation_downwards_mean`").alias("solar_radiation_mean"),
        F.col("w.`surface_solar_radiation_downwards_max`").alias("solar_radiation_max"),

        F.col("w.`total_cloud_cover_min`").alias("cloud_cover_min"),
        F.col("w.`total_cloud_cover_mean`").alias("cloud_cover_mean"),
        F.col("w.`total_cloud_cover_max`").alias("cloud_cover_max"),

        F.col("w.`utci_min`").alias("utci_min"),
        F.col("w.`utci_mean`").alias("utci_mean"),
        F.col("w.`utci_max`").alias("utci_max")
    )
)

In [0]:
silver.printSchema()

In [0]:
print(peak_long.columns)

In [0]:
print(electricity.columns)

In [0]:
#performing checks on silver layer
#checking for missing values
silver.select(
    F.count("*").alias("total_rows"),
    F.count("energy_demand_MU").alias("energy_rows"),
    F.count("peak_demand_MW").alias("peak_rows"),
    F.count("temperature_mean").alias("weather_rows")
).display()

In [0]:
#checking for state counts
silver.groupBy("state").count().display()

In [0]:
#check for duplicates
silver.groupBy(
    "date",
    "state"
).count().filter(
    F.col("count") > 1
).display()

In [0]:
#We see data Quality issues here we see we see duplicates